# Fundable API — quickstart walk

This notebook follows a single funding round through the dataset using **raw HTTP calls** — no `fundable` client import, just the `requests` library. It's the fastest way to see exactly what the API takes and returns.

The walk:

1. **Deals this week** — `POST /deals`
2. **The company that raised + the people who work there** — `GET /company`, `POST /people`
3. **The investors on that deal** — `GET /deals/{id}/investors`
4. **The people at the lead investment firm** — `POST /people`

> Convention: the collection endpoints (`/deals`, `/companies`, `/investors`, `/people`) are **POST** with a nested JSON body. Single-record and sub-resource reads (`/company`, `/deals/{id}/investors`) are **GET** with query params.

## Setup

Load your API key from a local `.env` file (`FUNDABLE_API_KEY=...`) and build the auth header. Every request below reuses `BASE_URL` and `HEADERS`.

In [ ]:
import os
from datetime import date, timedelta

import requests
from dotenv import load_dotenv

load_dotenv()  # reads FUNDABLE_API_KEY from a local .env file

API_KEY = os.environ["FUNDABLE_API_KEY"]
BASE_URL = "https://www.tryfundable.ai/api/v1"
HEADERS = {
    "Authorization": f"Bearer {API_KEY}",
    "Content-Type": "application/json",
}

## Step 1 — deals this week

`POST /deals` searches funding rounds. The body is grouped into sections (`deal`, `company`, `investors`); here we only filter by date. We ask for this week's rounds (Monday → today) sorted most-recent-first.

Each deal in the response carries `id`, `company_id`, `investor_ids`, and `angel_investor_ids` — IDs we'll resolve in the next steps. We walk the first deal that has a firm investor (some rounds are angel-only).

In [ ]:
today = date.today()
monday = today - timedelta(days=today.weekday())  # Monday of the current week

resp = requests.post(
    f"{BASE_URL}/deals",
    headers=HEADERS,
    json={
        "deal": {
            "date_start": monday.isoformat(),
            "date_end": today.isoformat(),
        },
        "page_size": 25,
        "sort_by": "most_recent_deal",
    },
    timeout=30,
)
resp.raise_for_status()
deals = resp.json()["data"]["deals"]

print(f"{len(deals)} deals from {monday} to {today}\n")

# Walk the first deal that has a firm investor, so every step below has
# something to show (some rounds are angel-only).
deal = next((d for d in deals if d.get("investor_ids")), deals[0])
amount = deal.get("total_round_raised")
amount_str = f"${amount:,.0f}" if amount else "undisclosed"
print("Walking this deal:")
print(f"  round       {deal.get('round_type')}")
print(f"  date        {deal.get('date')}")
print(f"  amount      {amount_str}")
print(f"  company_id  {deal['company_id']}")
print(
    f"  investors   {len(deal.get('investor_ids', []))} firm(s), "
    f"{len(deal.get('angel_investor_ids', []))} angel(s)"
)

## Step 2 — the company and its people

Two calls:

- `GET /company?id=...` returns the single company record (name, domain, …).
- `POST /people` with `person_type: "company"` and `company.ids` returns the people who work there. Any filter under `company.*` implicitly requires that to be the person's **current employer**, so this gives the company's current team. Each person has `name`, `title`, `linkedin_url`, and `is_founder`.

In [ ]:
company_id = deal["company_id"]

# Company detail (one record, by id)
resp = requests.get(
    f"{BASE_URL}/company",
    headers=HEADERS,
    params={"id": company_id},
    timeout=30,
)
resp.raise_for_status()
company = resp.json()["data"]["company"]
print(f"Company: {company.get('name')} ({company.get('domain')})\n")

# People at that company
resp = requests.post(
    f"{BASE_URL}/people",
    headers=HEADERS,
    json={
        "person_type": "company",
        "company": {"ids": [company_id]},
        "page_size": 10,
    },
    timeout=30,
)
resp.raise_for_status()
people = resp.json()["data"]["people"]
print(f"{len(people)} people at {company.get('name')}:")
for p in people:
    flag = " (founder)" if p.get("is_founder") else ""
    print(f"  - {p.get('name')} — {p.get('title') or 'n/a'}{flag}")

## Step 3 — the investors on the deal

`GET /deals/{id}/investors` returns the syndicate split into two arrays: `investors` (the **firms**, each with `id`, `name`, `lead_investor`, `domain`) and `angel_investors` (the **individuals**, surfaced separately). We'll carry the first firm into the last step.

In [ ]:
deal_id = deal["id"]

resp = requests.get(
    f"{BASE_URL}/deals/{deal_id}/investors",
    headers=HEADERS,
    timeout=30,
)
resp.raise_for_status()
data = resp.json()["data"]
firms = data["investors"]          # investment firms
angels = data["angel_investors"]   # individuals, surfaced separately

print(f"{len(firms)} firm(s) on this deal:")
for f in firms:
    lead = " [lead]" if f.get("lead_investor") else ""
    print(f"  - {f.get('name')} ({f.get('domain')}){lead}")

print(f"\n{len(angels)} angel(s):")
for a in angels:
    print(f"  - {a.get('name')}")

## Step 4 — the people at the investment firm

Finally, take the first firm from Step 3 and find its investors via `POST /people` with `person_type: "investor"` and `investor.ids`. Filters under `investor.*` implicitly require the person to be an investor, so this returns the firm's investment team.

In [ ]:
if not firms:
    print("This deal has no firm investors (angel-only) — nothing to look up.")
else:
    firm = firms[0]  # walk the first firm
    print(f"People at {firm.get('name')}:\n")

    resp = requests.post(
        f"{BASE_URL}/people",
        headers=HEADERS,
        json={
            "person_type": "investor",
            "investor": {"ids": [firm["id"]]},
            "page_size": 10,
        },
        timeout=30,
    )
    resp.raise_for_status()
    firm_people = resp.json()["data"]["people"]
    for p in firm_people:
        print(f"  - {p.get('name')} — {p.get('title') or 'n/a'}")

## Where to go next

That's the full walk: one deal → its company and team → its investors → the firm's people, all with plain `requests` calls.

The scripts in [`examples/`](examples/) do the same kinds of calls through the thin `fundable` client wrapper (`FundableClient`), which handles auth, body-building, and pagination for you. For the complete schema — every endpoint, filter, and response field — see the docs at <https://docs.tryfundable.ai/>.